# Damaged Books: huấn luyện YOLO11n trên Colab

Chạy lần lượt từ trên xuống. Chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU khả dụng).
Dataset được tải tự động từ GitHub Releases và kiểm SHA256. Có thể chọn tải ZIP thủ công.
Mỗi thành viên dùng một Colab/Drive riêng, chọn mã thí nghiệm ở bước 2.

5 lớp: `Barcode`, `Notch`, `Ripped`, `Spot`, `Wornout`. Barcode không phải lỗi.
253 ảnh train / 54 val / 54 test. 259 ảnh cách ly không dùng.
Notebook kiểm tra checksum và nhóm, train rồi báo cáo **validation**; không đánh giá test,
không chọn ngưỡng vận hành. Đánh giá test chỉ làm sau khi chốt model/ngưỡng bằng validation.

Dữ liệu: [Damaged Books v1, atcom21-gmail-com](https://universe.roboflow.com/atcom21-gmail-com/damaged-books/dataset/1), CC BY 4.0.
Ultralytics: AGPL-3.0 / giấy phép thương mại theo nhà cung cấp.
Chưa có bằng chứng chất lượng trên sách/camera của dự án.

In [ ]:
import sys, subprocess, importlib.metadata
ULTRALYTICS_VERSION = '8.4.208'
subprocess.run([sys.executable, '-m', 'pip', 'install',
                f'ultralytics=={ULTRALYTICS_VERSION}'], check=True)
import torch
assert torch.cuda.is_available(), 'Chưa có GPU: chọn Runtime → Change runtime type → GPU.'
assert importlib.metadata.version('ultralytics') == ULTRALYTICS_VERSION, 'Khởi động lại runtime rồi chạy lại.'
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)

## 1. Gói dữ liệu chung cho team
Mặc định ô dưới tải đúng `split-reviewed-v1.zip` từ GitHub Releases.
Nếu mạng tải lỗi, đổi `DATA_SOURCE = 'upload'` và chọn ZIP đã tải từ release.
Giữ nguyên checksum/version, không tự chia lại dataset cho từng người.

In [ ]:
from pathlib import Path
import hashlib, json, zipfile, math, shutil
from collections import Counter
from google.colab import files

EXPECTED_SHA256 = '0da9c0710a22cb3260506eb003692ee12ff322db70ddbca7f6687fccbadc56c1'
EXPECTED_VERSION = 'damaged-books-v1-five-8f203ddd57d4-077edf4cdcbd'
CLASSES = ['Barcode', 'Notch', 'Ripped', 'Spot', 'Wornout']
EXPECTED_COUNTS = {'train': 253, 'val': 54, 'test': 54}

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def validate_split(root):
    root = Path(root).resolve()
    manifest = json.loads((root / 'split_manifest.json').read_text(encoding='utf-8'))
    assert manifest['split_ready'] and manifest['classes'] == CLASSES
    assert manifest['dataset_version'] == EXPECTED_VERSION and manifest['seed'] == 42
    hashes = manifest['file_sha256']
    assert hashes, 'Thiếu checksum'
    for relative, digest in hashes.items():
        file = (root / relative).resolve()
        assert root in file.parents, 'Đường dẫn ngoài dataset'
        assert file.is_file() and sha256(file) == digest, f'Sai checksum: {relative}'
    seen_groups, seen_ids, seen_image_hashes = set(), set(), set()
    summary = {}
    for part, expected_count in EXPECTED_COUNTS.items():
        records = manifest['assignment'][part]
        ids = {r['image_id'] for r in records}
        groups = {r['group_id'] for r in records}
        assert len(records) == len(ids) == expected_count and all(groups)
        assert not (ids & seen_ids or groups & seen_groups), 'Trùng ID/nhóm giữa các tập'
        seen_ids |= ids
        seen_groups |= groups
        images = list((root / part / 'images').iterdir())
        labels = list((root / part / 'labels').iterdir())
        assert len(images) == len(ids) and {p.stem for p in images} == ids
        assert len(labels) == len(ids) and {p.stem for p in labels} == ids
        counts = Counter()
        part_hashes = set()
        for image in images:
            assert image.is_file() and image.suffix.lower() in {'.jpg', '.jpeg', '.png'}
            relative = image.relative_to(root).as_posix()
            assert relative in hashes
            part_hashes.add(hashes[relative])
        assert not part_hashes & seen_image_hashes, 'Trùng nội dung ảnh giữa các tập'
        seen_image_hashes |= part_hashes
        for label in labels:
            assert label.suffix == '.txt' and label.relative_to(root).as_posix() in hashes
            for line in label.read_text().splitlines():
                if not line.strip():
                    continue
                values = list(map(float, line.split()))
                assert len(values) == 5 and all(math.isfinite(v) for v in values)
                c, x, y, w, h = values
                assert c.is_integer() and 0 <= c < 5
                assert 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1
                counts[int(c)] += 1
        assert set(counts) == set(range(5)), f'{part} thiếu lớp'
        summary[part] = {'images': len(ids), 'groups': len(groups),
                         'boxes': {CLASSES[i]: counts[i] for i in range(5)}}
    assert not seen_ids & set(manifest['quarantined']), 'Ảnh cách ly lọt vào split'
    return manifest, summary

uploaded = None
DATA_SOURCE = 'release'  # 'release' hoặc 'upload'
DATASET_URL = ('https://github.com/nguyenhuunghia10t1-creator/'
               'pbl5-library-book-inspection/releases/download/'
               'damaged-books-reviewed-v1/split-reviewed-v1.zip')
if DATA_SOURCE == 'release':
    import urllib.request
    archive_path = Path('/content/split-reviewed-v1.zip')
    if not archive_path.exists() or sha256(archive_path) != EXPECTED_SHA256:
        temporary = archive_path.with_suffix('.download')
        try:
            request = urllib.request.Request(DATASET_URL, headers={'User-Agent': 'SmartVision-Colab'})
            with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            assert sha256(temporary) == EXPECTED_SHA256, 'ZIP tải về sai SHA256'
            temporary.replace(archive_path)
        finally:
            temporary.unlink(missing_ok=True)
elif DATA_SOURCE == 'upload':
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Chọn đúng một ZIP.'
    archive_path = Path(next(iter(uploaded)))
else:
    raise ValueError("DATA_SOURCE phải là 'release' hoặc 'upload'")
assert archive_path.suffix.lower() == '.zip' and sha256(archive_path) == EXPECTED_SHA256
del uploaded
dataset = Path('/content/damaged-books-reviewed-v1')
if not dataset.exists():
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (dataset / member.filename).resolve()
            assert target == dataset.resolve() or dataset.resolve() in target.parents
        assert archive.testzip() is None, 'ZIP hỏng'
        archive.extractall(dataset)
manifest, split_summary = validate_split(dataset)
print(json.dumps(split_summary, indent=2, ensure_ascii=False))
print('PASS: checksum, 5 lớp, số ảnh và không trùng nhóm/ảnh giữa các tập.')

## 2. Chọn thành viên, thí nghiệm và nơi lưu
Sửa `TEAM_MEMBER` thành mã của bạn (ví dụ `nghia`, `member02`).
Chia việc: người 1 chạy `n640`, người 2 chạy `s640`, người 3 chạy `n960`.
Nếu ít người, chạy lần lượt các cấu hình; mỗi cấu hình tạo lượt riêng.

- `n640`: YOLO11n, 640px — mốc so sánh ban đầu.
- `s640`: YOLO11s, 640px — thử model lớn hơn.
- `n960`: YOLO11n, 960px — thử độ phân giải đầu vào lớn hơn; ảnh nguồn vẫn chỉ 640px.

Giữ seed 42, optimizer và split giống nhau để so sánh. `BATCH_SIZE=16`;
nếu hết VRAM có thể giảm còn 8, và ghi nhận đây là cấu hình khác.
Mặc định lưu vào Drive; mỗi người kết nối Drive của mình. Chỉ chạy ô tạo lượt một lần.

In [ ]:
from datetime import datetime, timezone
import platform, re
TEAM_MEMBER = 'member01'
EXPERIMENT = 'n640'
BATCH_SIZE = 16
EXPERIMENTS = {
    'n640': {'weights': 'yolo11n.pt', 'imgsz': 640},
    's640': {'weights': 'yolo11s.pt', 'imgsz': 640},
    'n960': {'weights': 'yolo11n.pt', 'imgsz': 960},
}

def experiment_settings(member, experiment, batch):
    if not re.fullmatch(r'[A-Za-z0-9_-]{1,40}', member):
        raise ValueError('TEAM_MEMBER chỉ dùng chữ không dấu, số, _ hoặc - (1–40 ký tự)')
    if experiment not in EXPERIMENTS or type(batch) is not int or batch not in (8,16):
        raise ValueError('Chọn EXPERIMENT n640/s640/n960, BATCH_SIZE 8 hoặc 16')
    selected = EXPERIMENTS[experiment]
    config = dict(epochs=100, patience=20, batch=batch, imgsz=selected['imgsz'], seed=42,
                  deterministic=True, device=0, workers=2, optimizer='AdamW',
                  amp=True, save=True, save_period=10, plots=True)
    return selected['weights'], config

initial_weights, config = experiment_settings(TEAM_MEMBER, EXPERIMENT, BATCH_SIZE)
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    output_root = Path('/content/drive/MyDrive/SmartVision/damaged-books')
else:
    output_root = Path('/content/damaged-books-runs')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_id = f'{TEAM_MEMBER}-{EXPERIMENT}-{stamp}'
run = output_root / run_id
run.mkdir(parents=True, exist_ok=False)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')

# A separate YAML omits the test path, preventing accidental test evaluation here.
import yaml
runtime_yaml = run / 'train-val.yaml'
runtime_yaml.write_text(yaml.safe_dump({'path': str(dataset), 'train': 'train/images',
    'val': 'val/images', 'nc': 5, 'names': CLASSES}), encoding='utf-8')
shutil.copy2(dataset / 'split_manifest.json', run / 'split_manifest.json')
write_json(run / 'split-check.json', split_summary)
write_json(run / 'requested-config.json', config)
write_json(run / 'environment.json', {'python': platform.python_version(),
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0), 'ultralytics': ULTRALYTICS_VERSION,
    'team_member': TEAM_MEMBER, 'experiment': EXPERIMENT, 'run_id': run_id,
    'initial_weights': initial_weights, 'dataset_zip_sha256': EXPECTED_SHA256, 'dataset_version': manifest['dataset_version']})
(run / 'requirements-frozen.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print('Kết quả và checkpoint:', run)

## 3. Huấn luyện
Tối đa 100 epoch; dừng sớm sau 20 epoch không cải thiện validation.
Nếu GPU hết bộ nhớ, đổi `batch=8` ở ô cấu hình và tạo phiên chạy mới;
cấu hình thực tế sẽ được lưu. Không chỉnh tham số dựa trên tập test.
Nếu Colab ngắt, `train/weights/last.pt` trên Drive là checkpoint;
giữ lại đường dẫn đó để tiếp tục phiên sau, không ghi đè bằng một lần train mới.

In [ ]:
from ultralytics import YOLO
assert not (run / 'train').exists(), 'Phiên này đã bắt đầu. Không ghi đè; kiểm checkpoint hoặc tạo phiên mới.'
model = YOLO(initial_weights)
write_json(run / 'initial-weights.json', {'name': initial_weights, 'sha256': sha256(initial_weights)})
try:
    model.train(data=str(runtime_yaml), project=str(run), name='train', exist_ok=False, **config)
except Exception as error:
    write_json(run / 'training-status.json', {'status': 'failed', 'error': str(error)})
    raise
best = run / 'train/weights/best.pt'
assert best.is_file(), 'Chưa có best.pt; kiểm tra log huấn luyện.'
write_json(run / 'training-status.json', {'status': 'completed', 'best_sha256': sha256(best)})
print('Model tốt nhất trên validation:', best)

## 4. Báo cáo validation và tải kết quả
Precision/recall ở đây là thống kê Ultralytics trên validation, chưa phải ngưỡng vận hành từng lớp.
ZIP chứa best.pt, checkpoint, cấu hình thực tế (`train/args.yaml`), lịch sử epoch,
confusion matrix, bảng lớp, phiên bản dữ liệu và thư viện. Test vẫn chưa đánh giá.

In [ ]:
best = run / 'train/weights/best.pt'
trained = YOLO(str(best))
assert [trained.names[i] for i in range(5)] == CLASSES, 'Sai bảng lớp model'
metrics = trained.val(data=str(runtime_yaml), split='val', imgsz=config['imgsz'], batch=config['batch'],
                      device=0, workers=2, plots=True, project=str(run), name='validation')
per_class = {}
for class_id in range(5):
    positions = list(map(int, metrics.box.ap_class_index))
    if class_id in positions:
        j = positions.index(class_id)
        per_class[CLASSES[class_id]] = {'precision': float(metrics.box.p[j]),
            'recall': float(metrics.box.r[j]), 'ap50': float(metrics.box.ap50[j]),
            'ap50_95': float(metrics.box.ap[j])}
    else:
        per_class[CLASSES[class_id]] = {'precision': None, 'recall': None,
            'ap50': None, 'ap50_95': None, 'status': 'not_reported'}
write_json(run / 'validation-metrics.json', {
    'split': 'val', 'aggregate': {k: float(v) for k,v in metrics.results_dict.items()},
    'per_class': per_class, 'speed_ms_on_colab': metrics.speed,
    'note': 'Ultralytics summary; not per-class deployment thresholds or local CPU latency'})
write_json(run / 'model-metadata.json', {'schema_version': 1, 'classes': CLASSES,
    'model_version': 'yolo11-five-' + run_id,
    'run_id': run_id, 'team_member': TEAM_MEMBER, 'experiment': EXPERIMENT, 'initial_weights': initial_weights, 'dataset_version': manifest['dataset_version'],
    'weights_sha256': sha256(best), 'thresholds': None, 'threshold_status': 'not_calibrated',
    'test_evaluated': False, 'camera_domain_accepted': False, 'scope': 'visible_face_only',
    'dataset_license': manifest['license'], 'dataset_source': manifest['source']})
# One comparable validation summary per run. No test measurements are included.
import csv
row = {'run_id': run_id, 'team_member': TEAM_MEMBER, 'experiment': EXPERIMENT,
       'initial_weights': initial_weights, 'dataset_version': manifest['dataset_version'],
       'dataset_zip_sha256': EXPECTED_SHA256, 'split': 'val', 'test_evaluated': False,
       'imgsz': config['imgsz'], 'batch': config['batch'], 'seed': config['seed'],
       'optimizer': config['optimizer'], 'gpu': torch.cuda.get_device_name(0),
       'ultralytics': ULTRALYTICS_VERSION, 'best_sha256': sha256(best),
       'weights_bytes': best.stat().st_size}
for name, value in metrics.results_dict.items():
    row[name] = float(value)
for name, values in per_class.items():
    for metric_name, value in values.items():
        row[name + '_' + metric_name] = value
with (run / 'experiment-summary.csv').open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(row))
    writer.writeheader()
    writer.writerow(row)

archive = shutil.make_archive(str(run.parent / (run.name + '-artifacts')), 'zip', run)
print(json.dumps(per_class, indent=2, ensure_ascii=False))
print('Giữ ZIP này để chọn ngưỡng bằng validation và đánh giá test ở bước sau:', archive)
files.download(archive)

## Team so sánh kết quả
Mỗi người gửi ZIP kết quả và `experiment-summary.csv` của lượt mình vào nơi team thống nhất.
So sánh cùng dataset/version và thư viện; xem precision/recall của Ripped cùng mAP50–95,
confusion matrix và độ lớn model. Precision/recall ở đây là thống kê Ultralytics,
chưa phải metric tại ngưỡng vận hành đã chọn.

Không dùng test để xếp hạng các lượt thử; không kết luận chỉ từ một con số.
Sau khi thu hẹp ứng viên, chọn ngưỡng trên validation (F2 Ripped, F1 lớp khác),
chốt model/cấu hình rồi đánh giá test cuối. Muốn so CPU, đo các model trên cùng máy.
Phải thử ảnh sách trường/camera dự án trước khi nghiệm thu. Chưa tích hợp app trong notebook này.